In [1]:
!pip install google-genai torch torchvision opencv-python-headless



[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# Install the CUDA 12 specific version of cuCIM
!pip install cucim-cu12

# Install CuPy for CUDA 12
!pip install cupy-cuda12x

# Install the required image format plugins
!pip install pylibcucim-cu12

  Using cached cucim_cu12-26.2.0.tar.gz (3.8 kB)
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'error'


  error: subprocess-exited-with-error
  
  × Preparing metadata (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> [67 lines of output]
      INFO:wheel-stub:Testing wheel cucim_cu12-26.2.0-cp310-cp310-manylinux_2_26_aarch64.manylinux_2_28_aarch64.whl against tag cp310-cp310-manylinux_2_26_aarch64
      INFO:wheel-stub:Testing wheel cucim_cu12-26.2.0-cp310-cp310-manylinux_2_26_aarch64.manylinux_2_28_aarch64.whl against tag cp310-cp310-manylinux_2_28_aarch64
      INFO:wheel-stub:Testing wheel cucim_cu12-26.2.0-cp310-cp310-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl against tag cp310-cp310-manylinux_2_27_x86_64
      INFO:wheel-stub:Testing wheel cucim_cu12-26.2.0-cp310-cp310-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl against tag cp310-cp310-manylinux_2_28_x86_64
      INFO:wheel-stub:Testing wheel cucim_cu12-26.2.0-cp311-cp311-manylinux_2_26_aarch64.manylinux_2_28_aarch64.whl against tag cp311-cp311-manylinux_2_28_aarch64
      INFO:wheel-stub:Testing wheel cuci


[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
from IPython.display import display, Image, Audio
import openai
import cv2  # We're using OpenCV to read video, to install !pip install opencv-python
import base64
import time
from openai import OpenAI
import os
import requests
import torch

In [4]:
# In a Python cell
!lspci | grep -i nvidia
!nvidia-smi
!nvcc --version
!which python
!python --version

'lspci' is not recognized as an internal or external command,
operable program or batch file.


Tue May  5 20:09:21 2026       
+---------------------------------------------------------------------------------------+
| NVIDIA-SMI 539.28                 Driver Version: 539.28       CUDA Version: 12.2     |
|-----------------------------------------+----------------------+----------------------+
| GPU  Name                     TCC/WDDM  | Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |         Memory-Usage | GPU-Util  Compute M. |
|                                         |                      |               MIG M. |
|=========================================+======================+======================|
|   0  NVIDIA H100 NVL              TCC   | 00000000:01:00.0 Off |                    0 |
| N/A   27C    P0              60W / 400W |      1MiB / 95830MiB |      0%      Default |
|                                         |                      |             Disabled |
+-----------------------------------------+----------------------+--

'which' is not recognized as an internal or external command,
operable program or batch file.


Python 3.10.11


In [5]:
!nvidia-smi

Tue May  5 20:09:22 2026       
+---------------------------------------------------------------------------------------+
| NVIDIA-SMI 539.28                 Driver Version: 539.28       CUDA Version: 12.2     |
|-----------------------------------------+----------------------+----------------------+
| GPU  Name                     TCC/WDDM  | Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |         Memory-Usage | GPU-Util  Compute M. |
|                                         |                      |               MIG M. |
|=========================================+======================+======================|
|   0  NVIDIA H100 NVL              TCC   | 00000000:01:00.0 Off |                    0 |
| N/A   27C    P0              60W / 400W |      1MiB / 95830MiB |      0%      Default |
|                                         |                      |             Disabled |
+-----------------------------------------+----------------------+--

In [6]:
import os

# Load Gemini API key from local file
API_KEY_PATH = r"C:\Opeyemi\PROMPTS\API-KEYS\ope-gemi.txt"
with open(API_KEY_PATH, "r") as f:
    GEMINI_API_KEY = f.read().strip()

os.environ["GOOGLE_API_KEY"] = GEMINI_API_KEY
os.environ["GEMINI_API_KEY"] = GEMINI_API_KEY

print("✓ Gemini API key loaded")
print(f"  Key file: {API_KEY_PATH}")
print(f"  Key length: {len(GEMINI_API_KEY)} chars")


✓ Gemini API key loaded
  Key file: C:\Opeyemi\PROMPTS\API-KEYS\ope-gemi.txt
  Key length: 53 chars


## Dataset: UCF-Crime — Pre-Extracted Frames

**Frames must be pre-extracted before running any technique.**

Expected folder structure (either layout supported):
```
# Sub-folder layout (recommended):
FRAMES_DIR/
  Abuse/
    Abuse001_x264/   frame_000000.png  frame_000030.png  ...
    Abuse002_x264/   ...
  Arrest/ Arson/ Assault/ Burglary/ Explosion/ Fighting/
  RoadAccidents/ Robbery/ Shooting/ Shoplifting/ Stealing/ Vandalism/
```

**Configuration** (top of each technique cell):
```python
DATA_DIR   = r"C:\Opeyemi\PROMPTS\UCF-Data"   # original videos (reference only)

FRAMES_DIR = r"C:\Opeyemi\PROMPTS\FRAMES"      # pre-extracted frames

FRAME_INTERVAL = 1   # 1 = all frames | 2 = every 2nd frame | etc.
```


# Shared Utilities: Checkpoint System + NeurIPS Compute Tracker
Run this cell before any technique cell. It provides:
- **CheckpointManager** — saves per-chunk results to disk; resumes after network failures
- **ComputeTracker** — accumulates token estimates, API call stats, timing for NeurIPS reporting

In [7]:
# =============================================================================
# SHARED UTILITIES: Checkpoint System + NeurIPS Compute Tracker
# Run this cell BEFORE any technique cell.
# =============================================================================
import os, json, time


class CheckpointManager:
    """
    Persists per-chunk API responses to disk so that a network failure or
    kernel restart never causes already-finished work to be repeated.

    Layout:
      <save_dir>/_checkpoints/<Technique>_<video_key>.json
      {
        "complete": false,
        "chunks":   {"chunk_key": "<response_text>", ...},
        "result":   null          # filled when the video is fully processed
      }
    """
    def __init__(self, save_dir, technique_name):
        self.cp_dir    = os.path.join(save_dir, "_checkpoints")
        self.technique = technique_name
        os.makedirs(self.cp_dir, exist_ok=True)

    def _path(self, vk):
        safe = vk.replace("/", "_").replace(" ", "_")[:120]
        return os.path.join(self.cp_dir, f"{self.technique}_{safe}.json")

    def _read(self, vk):
        p = self._path(vk)
        if os.path.exists(p):
            with open(p) as f:
                return json.load(f)
        return {"complete": False, "chunks": {}, "result": None}

    def _write(self, vk, state):
        with open(self._path(vk), "w") as f:
            json.dump(state, f, indent=2)

    # ── video-level ───────────────────────────────────────────────────────────
    def is_video_complete(self, vk):
        return self._read(vk).get("complete", False)

    def get_video_result(self, vk):
        return self._read(vk).get("result")

    def mark_video_complete(self, vk, result):
        s = self._read(vk)
        s.update({"complete": True, "result": result,
                  "done_at": time.strftime("%Y%m%d_%H%M%S")})
        self._write(vk, s)

    # ── chunk-level ───────────────────────────────────────────────────────────
    def is_chunk_done(self, vk, ck):
        return ck in self._read(vk).get("chunks", {})

    def get_chunk(self, vk, ck):
        return self._read(vk).get("chunks", {}).get(ck)

    def save_chunk(self, vk, ck, text):
        s = self._read(vk)
        s.setdefault("chunks", {})[ck] = text
        s["last_updated"] = time.strftime("%Y%m%d_%H%M%S")
        self._write(vk, s)


class ComputeTracker:
    """
    Accumulates per-call statistics for NeurIPS-style compute reporting.

    Token estimation methodology
    ─────────────────────────────
    Text input  : characters / 4  (standard English approximation)
    Image input : 258 tokens / image  (Gemini vision billing constant)
    Output      : usageMetadata.candidatesTokenCount when present,
                  otherwise output_chars / 4
    """
    _IMG_TOK = 258

    def __init__(self, model_name, technique_name):
        self.model     = model_name
        self.technique = technique_name
        self._t0       = time.time()
        self.calls = self.ok = self.failed = 0
        self.in_tok = self.out_tok = self.img_count = 0
        self.latencies = []
        self.temps     = set()
        self.videos = self.frames = 0

    def record(self, *, success, prompt_chars, n_images, out_tok, temp, latency):
        self.calls += 1
        if success:
            self.ok += 1
        else:
            self.failed += 1
        self.in_tok    += max(prompt_chars, 0) // 4
        self.img_count += n_images
        self.out_tok   += out_tok
        self.latencies.append(latency)
        self.temps.add(round(float(temp), 2))

    def record_video(self, n_frames):
        self.videos += 1
        self.frames += n_frames

    def report(self):
        elapsed  = time.time() - self._t0
        avg_lat  = sum(self.latencies) / max(len(self.latencies), 1)
        img_tok  = self.img_count * self._IMG_TOK
        total    = self.in_tok + img_tok + self.out_tok
        return {
            "NeurIPS_Compute_Report": {
                "model"      : self.model,
                "technique"  : self.technique,
                "wall_clock" : {
                    "seconds": round(elapsed, 2),
                    "hours"  : round(elapsed / 3600, 5)
                },
                "api_calls"  : {
                    "total"         : self.calls,
                    "successful"    : self.ok,
                    "failed"        : self.failed,
                    "avg_latency_s" : round(avg_lat, 3)
                },
                "token_budget": {
                    "text_input_est"  : self.in_tok,
                    "image_input_est" : img_tok,
                    "output"          : self.out_tok,
                    "grand_total_est" : total,
                    "images_sent"     : self.img_count,
                    "methodology"     : (
                        "text_input: chars/4 | "
                        "image_input: 258 tok/image (Gemini vision pricing) | "
                        "output: usageMetadata.candidatesTokenCount when available"
                    )
                },
                "data_processed"  : {"videos": self.videos, "frames": self.frames},
                "hyperparameters" : {
                    "temperatures"      : sorted(self.temps),
                    "max_output_tokens" : 4096,
                    "top_p"             : 0.8,
                    "top_k"             : 10,
                    "chunk_size"        : 10
                },
                "reproducibility": {
                    "model_string": self.model,
                    "api_endpoint": "Gemini Developer API (google-genai SDK)",
                    "api_version" : "vertex-ai-genai",
                    "random_seed" : (
                        "N/A - Gemini REST API does not expose a seed parameter. "
                        "At temperature=0.1, outputs are near-deterministic; "
                        "minor token-sampling variation is possible across runs."
                    )
                }
            }
        }


print("Checkpoint and compute-tracking utilities loaded.")


Checkpoint and compute-tracking utilities loaded.


# Sequential Prompting

Sequential Prompting Implementation

Key Features:
- Processes all frames in chunks
- Uses a sequence of 5 prompts that build on each other:
  1. High-level scene description
  2. People and actions identification
  3. Criminal activity analysis
  4. Objects/items involved
  5. Chronological timeline of events
- Includes a final synthesis that integrates all prior responses

Implementation Highlights:
- Each step references prior step's response (true sequential conditioning)
- Each chunk gets analyzed separately, with results combined before the next step
- Conversation history carries forward across all 5 steps


In [8]:
import os
import json
import base64
import re
import requests
from google import genai
from google.genai import types
import time
from datetime import datetime
from collections import defaultdict

import threading
from concurrent.futures import ThreadPoolExecutor, as_completed
# Running on local machine
# Ensure Gemini API key is set (run cell 5 first)
if "GOOGLE_API_KEY" not in os.environ:
    print("⚠ WARNING: Run the credentials cell (cell 5) first!")
    print("  GOOGLE_API_KEY not set")

# Configuration
DATA_DIR   = "C:\\Opeyemi\\PROMPTS\\UCF-Data"   # root with crime-type subfolders
FRAMES_DIR = r"C:\\Opeyemi\\PROMPTS\\FRAMES"  # pre-extracted frames
SAVE_DIR = "C:\\Opeyemi\\PROMPTS\\RESULTS\\GEMINI\\SEQUENTIAL"
FRAME_INTERVAL = 1  # Sample every Nth frame (1 = all frames)
MAX_WORKERS    = 4    # parallel videos processed at once (Gemini quotas; raise carefully)
BATCH_SIZE     = 20   # frames per API call (must be defined per-cell so cells run independently)

class SequentialPromptingAnalyzer:
    def __init__(self):
        self.model_name = "gemini-3.1-pro-preview"
        self.client = genai.Client(
            api_key=os.environ["GOOGLE_API_KEY"],
        )
        self.save_dir = SAVE_DIR
        self.chunk_size = BATCH_SIZE
        os.makedirs(self.save_dir, exist_ok=True)
        self._ckpt = CheckpointManager(self.save_dir, type(self).__name__)
        self._comp = ComputeTracker(self.model_name, type(self).__name__)
        # Sequential prompting flow - each prompt builds on previous responses
        self.prompt_sequence = [
            "What's happening in these frames? Describe the scene at a high level.",
            "Based on what you observed in your previous response, who are the main people in the scene and what are they doing?",
            "Looking at the actions you described, do you observe any potential criminal activities? If so, describe them in detail.",
            "Based on your crime analysis, what objects or items are involved in the incident?",
            "Considering all your observations, create a chronological timeline of events shown in these frames."
        ]


    def _call_gemini_sdk_from_payload(self, payload):
        """Convert REST API payload to genai SDK call and return REST-compatible response dict."""
        try:
            contents_list = payload.get("contents", [])
            gen_config = payload.get("generationConfig", {})

            sdk_parts = []
            for content_block in contents_list:
                for part in content_block.get("parts", []):
                    if "text" in part:
                        sdk_parts.append(types.Part.from_text(text=part["text"]))
                    elif "inline_data" in part:
                        mime = part["inline_data"].get("mime_type", "image/png")
                        data_bytes = base64.b64decode(part["inline_data"]["data"])
                        sdk_parts.append(types.Part.from_bytes(data=data_bytes, mime_type=mime))

            sdk_content = types.Content(role="user", parts=sdk_parts)

            config = types.GenerateContentConfig(
                temperature=gen_config.get("temperature", 0.1),
                max_output_tokens=gen_config.get("maxOutputTokens", 4096),
                top_p=gen_config.get("topP", 0.8),
                top_k=gen_config.get("topK", 10),
            )

            response = self.client.models.generate_content(
                model=self.model_name,
                contents=sdk_content,
                config=config,
            )

            if response.text:
                return {
                    "candidates": [{
                        "content": {
                            "parts": [{"text": response.text}]
                        }
                    }],
                    "usageMetadata": {
                        "promptTokenCount": getattr(response.usage_metadata, 'prompt_token_count', 0) if response.usage_metadata else 0,
                        "candidatesTokenCount": getattr(response.usage_metadata, 'candidates_token_count', 0) if response.usage_metadata else 0,
                    }
                }
            else:
                return {"error": "No text in response", "candidates": []}

        except Exception as e:
            error_msg = str(e)
            print(f"API Error: {error_msg}")
            if "429" in error_msg or "RESOURCE_EXHAUSTED" in error_msg or "quota" in error_msg.lower():
                print("Rate limited - waiting 60 seconds...")
                time.sleep(60)
                return self._call_gemini_sdk_from_payload(payload)
            return {"error": error_msg}


    def convert_conversation_to_gemini_format(self, conversation, new_user_content):
        """Convert conversation history to Gemini format"""
        contents = []

        for i in range(1, len(conversation)):
            message = conversation[i]
            if message["role"] == "user":
                if isinstance(message["content"], list):
                    parts = []
                    for content_item in message["content"]:
                        if content_item["type"] == "text":
                            parts.append({"text": content_item["text"]})
                        elif content_item["type"] == "image_url":
                            data_url = content_item["image_url"]["url"]
                            if data_url.startswith("data:"):
                                mime_type, base64_data = data_url.split(",", 1)
                                mime_type = mime_type.split(":")[1].split(";")[0]
                                parts.append({
                                    "inline_data": {
                                        "mime_type": mime_type,
                                        "data": base64_data
                                    }
                                })
                    contents.append({"role": "user", "parts": parts})
                else:
                    contents.append({"role": "user", "parts": [{"text": message["content"]}]})
            elif message["role"] == "assistant":
                contents.append({"role": "model", "parts": [{"text": message["content"]}]})

        if isinstance(new_user_content, list):
            parts = []
            for content_item in new_user_content:
                if content_item["type"] == "text":
                    parts.append({"text": content_item["text"]})
                elif content_item["type"] == "image_url":
                    data_url = content_item["image_url"]["url"]
                    if data_url.startswith("data:"):
                        mime_type, base64_data = data_url.split(",", 1)
                        mime_type = mime_type.split(":")[1].split(";")[0]
                        parts.append({
                            "inline_data": {
                                "mime_type": mime_type,
                                "data": base64_data
                            }
                        })
            contents.append({"role": "user", "parts": parts})
        else:
            contents.append({"role": "user", "parts": [{"text": new_user_content}]})

        return contents

    def process_frames_with_sequential_prompts(self, frames_data, video_id, crime_type):
        """Process frames with sequential prompting approach"""
        frame_names = list(frames_data.keys())

        def extract_frame_number(filename):
            try:
                if '_frame_' in filename:
                    parts = filename.split('_frame_')
                    if len(parts) > 1:
                        number_part = parts[1].split('.')[0]
                        return int(number_part)
                elif 'frame' in filename.lower():
                    import re
                    numbers = re.findall(r'\d+', filename)
                    if numbers:
                        return int(numbers[-1])
            except Exception as e:
                print(f"Error extracting frame number from {filename}: {str(e)}")
                return 0

        sorted_frames = sorted(frame_names, key=extract_frame_number)
        frame_data = [frames_data[frame_name] for frame_name in sorted_frames if frame_name in frames_data and frames_data[frame_name]]

        if not frame_data:
            return {"error": "No valid frames available for analysis"}

        total_frames = len(frame_data)
        print(f"Processing all {total_frames} frames for sequential analysis")

        chunk_size = BATCH_SIZE
        frame_chunks = [frame_data[i:i+chunk_size] for i in range(0, total_frames, chunk_size)]
        print(f"Split into {len(frame_chunks)} chunks of approximately {chunk_size} frames each")

        conversation = [
            {
                "role": "system",
                "content": "You are analyzing video frames showing a potential crime scene. Provide detailed observations based on what you see."
            }
        ]

        sequence_results = {}
        all_chunk_responses = {}

        for step, prompt in enumerate(self.prompt_sequence, 1):
            print(f"Processing sequential prompt {step}/{len(self.prompt_sequence)}")

            step_responses = []

            for chunk_idx, chunk in enumerate(frame_chunks):
                print(f"  Processing chunk {chunk_idx+1}/{len(frame_chunks)} for step {step}...")

                _vk_inner = f"{crime_type}_{video_id}"
                _ck = f"step_{step}_chunk_{chunk_idx}"
                if self._ckpt.is_chunk_done(_vk_inner, _ck):
                    print(f"  [CHECKPOINT] Step {step} chunk {chunk_idx+1} already done")
                    step_responses.append(self._ckpt.get_chunk(_vk_inner, _ck))
                    continue

                if step == 1:
                    chunk_conversation = [conversation[0]]
                else:
                    chunk_conversation = [conversation[0]]
                    for prev_step in range(1, step):
                        chunk_conversation.append({
                            "role": "user",
                            "content": self.prompt_sequence[prev_step-1]
                        })
                        chunk_conversation.append({
                            "role": "assistant",
                            "content": all_chunk_responses[f"Step {prev_step}"]
                        })

                user_message_content = [
                    {
                        "type": "text",
                        "text": f"{prompt} (Analyzing frames {chunk_idx*chunk_size+1}-{min((chunk_idx+1)*chunk_size, total_frames)} of {total_frames})"
                    }
                ]

                for frame in chunk:
                    mime_type = "image/png"
                    user_message_content.append({
                        "type": "image_url",
                        "image_url": {
                            "url": f"data:{mime_type};base64,{frame}",
                            "detail": "high"
                        }
                    })

                gemini_contents = self.convert_conversation_to_gemini_format(chunk_conversation, user_message_content)

                payload = {
                    "contents": gemini_contents,
                    "generationConfig": {
                        "temperature": 0.1,
                        "maxOutputTokens": 4096,
                        "topP": 0.8,
                        "topK": 10
                    }
                }

                try:
                    print(f"    Sending request to Gemini for chunk {chunk_idx+1}...")
                    _t0_req = time.time()
                    response = self._call_gemini_sdk_from_payload(payload)
                    _latency = time.time() - _t0_req

                    if response is None or (isinstance(response, dict) and "error" in response):
                        error_detail = response.json() if response.text else response.text
                        print(f"    API Error {response.status_code}: {error_detail}")
                        step_responses.append(f"Error processing chunk {chunk_idx+1}: {error_detail}")
                        continue

                    result = response
                    if isinstance(result, dict) and "candidates" in result and result["candidates"]:
                        if "content" in result["candidates"][0] and "parts" in result["candidates"][0]["content"]:
                            assistant_response = result["candidates"][0]["content"]["parts"][0]["text"]
                            print(f"    Received response for chunk {chunk_idx+1}")
                            step_responses.append(assistant_response)
                            self._ckpt.save_chunk(_vk_inner, _ck, assistant_response)
                            _usage = result.get("usageMetadata", {})
                            _out_tok = _usage.get("candidatesTokenCount", len(assistant_response)//4)
                            _n_imgs = sum(1 for p in gemini_contents[-1].get("parts",[]) if "inline_data" in p)
                            self._comp.record(success=True, prompt_chars=len(str(payload)),
                                              n_images=_n_imgs, out_tok=_out_tok,
                                              temp=payload.get("generationConfig",{}).get("temperature",0.1),
                                              latency=_latency)
                        else:
                            step_responses.append(f"No content in response for chunk {chunk_idx+1}")
                    else:
                        step_responses.append(f"No candidates in response for chunk {chunk_idx+1}")

                except Exception as e:
                    print(f"    Error in chunk {chunk_idx+1} for step {step}: {str(e)}")
                    step_responses.append(f"Error processing chunk {chunk_idx+1}: {str(e)}")

                print(f"    Waiting 3 seconds before next request...")
                time.sleep(3)

            combined_response = "\n\n=== NEXT CHUNK ===\n\n".join(step_responses)

            sequence_results[f"Step {step}"] = {
                "prompt": prompt,
                "response": combined_response
            }

            all_chunk_responses[f"Step {step}"] = combined_response

            conversation.append({
                "role": "user",
                "content": prompt
            })
            conversation.append({
                "role": "assistant",
                "content": combined_response
            })

            timestamp = time.strftime("%Y%m%d_%H%M%S")
            step_result = {
                f"Step {step}": sequence_results[f"Step {step}"]
            }
            self.save_results(step_result, f"{crime_type}_{video_id}_sequential_step{step}_{timestamp}.json")
            print(f"  Step {step} results saved.")

            print(f"  Waiting 5 seconds before next step...")
            time.sleep(5)

        synthesis_prompt = "Based on all your previous analyses of ALL frame chunks, provide a comprehensive final assessment of the entire video. Summarize what crime appears to be taking place, who is involved, and how events unfolded across all the frames."

        gemini_contents = self.convert_conversation_to_gemini_format(conversation, synthesis_prompt)

        payload = {
            "contents": gemini_contents,
            "generationConfig": {
                "temperature": 0.1,
                "maxOutputTokens": 4096,
                "topP": 0.8,
                "topK": 10
            }
        }

        try:
            print("Performing final synthesis of all analyses...")
            _t0_synth = time.time()
            response = self._call_gemini_sdk_from_payload(payload)
            _lat_synth = time.time() - _t0_synth

            if response is not None and not (isinstance(response, dict) and "error" in response):
                result = response
                if isinstance(result, dict) and "candidates" in result and result["candidates"]:
                    if "content" in result["candidates"][0] and "parts" in result["candidates"][0]["content"]:
                        assistant_response = result["candidates"][0]["content"]["parts"][0]["text"]
                        print("Synthesis complete!")
                        _usage_s = result.get("usageMetadata", {})
                        _out_tok_s = _usage_s.get("candidatesTokenCount", len(assistant_response)//4)
                        self._comp.record(success=True, prompt_chars=len(str(payload)),
                                          n_images=0, out_tok=_out_tok_s,
                                          temp=0.1, latency=_lat_synth)

                        sequence_results["Final Synthesis"] = {
                            "prompt": synthesis_prompt,
                            "response": assistant_response
                        }

                        timestamp = time.strftime("%Y%m%d_%H%M%S")
                        synthesis_result = {
                            "Final Synthesis": sequence_results["Final Synthesis"]
                        }
                        self.save_results(synthesis_result, f"{crime_type}_{video_id}_sequential_synthesis_{timestamp}.json")
                        print("Synthesis results saved.")
                else:
                    print("No content in synthesis response")
            else:
                error_detail = response.json() if response.text else response.text
                print(f"API Error in synthesis: {response.status_code}: {error_detail}")

        except Exception as e:
            print(f"Error in final synthesis: {str(e)}")

        return {
            "sequential_results": sequence_results,
            "frames_used": total_frames,
            "chunks_processed": len(frame_chunks),
            "frames_per_chunk": chunk_size,
            "model_used": self.model_name,
            "crime_type": crime_type,
            "prompting_technique": "SEQUENTIAL_PROMPTING",
            "timestamp": time.strftime("%Y%m%d_%H%M%S")
        }

    def save_results(self, results, filename):
        """Save results to a file"""
        filepath = os.path.join(self.save_dir, filename)
        with open(filepath, 'w') as f:
            json.dump(results, f, indent=2)
        print(f"Results saved to: {filepath}")

    def analyze_frames(self, frames_data, video_id, crime_type):
        """Analyze frames with sequential prompting"""
        try:
            print(f"\n=== ANALYZING VIDEO: {video_id} ({crime_type}) WITH SEQUENTIAL PROMPTING ===")
            print(f"Total frames loaded: {len(frames_data)}")
            print(f"Using model: {self.model_name}")

            _vk = f"{crime_type}_{video_id}"
            if self._ckpt.is_video_complete(_vk):
                print(f"  [CHECKPOINT] {video_id} already complete, loading from disk")
                return self._ckpt.get_video_result(_vk)

            timestamp = time.strftime("%Y%m%d_%H%M%S")
            results = self.process_frames_with_sequential_prompts(frames_data, video_id, crime_type)

            self.save_results(results, f"{crime_type}_{video_id}_sequential_complete_{timestamp}.json")
            self._ckpt.mark_video_complete(_vk, results)
            self._comp.record_video(len(frames_data))
            print(f"Complete sequential analysis for {video_id} ({crime_type}) saved.")

            return results

        except Exception as e:
            print(f"Error in sequential analysis: {str(e)}")
            return {"error": str(e)}

def discover_all_videos_and_frames(frames_dir):
    """
    Discover pre-extracted frames from FRAMES_DIR.
    Expected layout:
      FRAMES_DIR/<CrimeType>/<VideoID>/frame_000000.png ...
    or (flat):
      FRAMES_DIR/<CrimeType>/<VideoID_frame_XXXXXX>.png ...
    Both layouts are supported automatically.
    """
    print(f"\n=== DISCOVERING PRE-EXTRACTED FRAMES ===")
    print(f"Scanning: {frames_dir}")

    IMAGE_EXTS = {".png", ".jpg", ".jpeg", ".bmp"}
    all_videos = {}

    try:
        # ONLY process the 'Stealing' folder (case-insensitive match)
        TARGET_CRIME = "Stealing"
        crime_types = sorted([
            d for d in os.listdir(frames_dir)
            if os.path.isdir(os.path.join(frames_dir, d))
            and d.lower() == TARGET_CRIME.lower()
        ])
        print(f"Crime-type folders to process (filtered to '{TARGET_CRIME}'): {crime_types}")
        if not crime_types:
            print(f"⚠ WARNING: No folder matching '{TARGET_CRIME}' found in {frames_dir}")

        for crime_type in crime_types:
            crime_dir = os.path.join(frames_dir, crime_type)

            sub_dirs = [
                d for d in os.listdir(crime_dir)
                if os.path.isdir(os.path.join(crime_dir, d))
            ]

            if sub_dirs:
                for video_id in sorted(sub_dirs):
                    video_frame_dir = os.path.join(crime_dir, video_id)
                    frames = sorted([
                        f for f in os.listdir(video_frame_dir)
                        if os.path.splitext(f.lower())[1] in IMAGE_EXTS
                    ])
                    if frames:
                        vk = f"{crime_type}_{video_id}"
                        all_videos[vk] = {
                            "crime_type" : crime_type,
                            "video_id"   : video_id,
                            "frames"     : frames,
                            "crime_dir"  : video_frame_dir,
                            "video_path" : None,
                        }
                        print(f"  {crime_type}/{video_id}: {len(frames)} frames")
            else:
                all_files = sorted([
                    f for f in os.listdir(crime_dir)
                    if os.path.splitext(f.lower())[1] in IMAGE_EXTS
                ])
                video_groups = defaultdict(list)
                for fname in all_files:
                    base = os.path.splitext(fname)[0]
                    if "_frame_" in base:
                        vid_id = base.split("_frame_")[0]
                    else:
                        vid_id = re.sub(r"_?\d+$", "", base) or base
                    video_groups[vid_id].append(fname)

                for video_id, frames in sorted(video_groups.items()):
                    if frames:
                        vk = f"{crime_type}_{video_id}"
                        all_videos[vk] = {
                            "crime_type" : crime_type,
                            "video_id"   : video_id,
                            "frames"     : sorted(frames),
                            "crime_dir"  : crime_dir,
                            "video_path" : None,
                        }
                        print(f"  {crime_type}/{video_id}: {len(frames)} frames")

    except Exception as e:
        print(f"Error scanning {frames_dir}: {e}")

    print(f"\nTotal videos found: {len(all_videos)}")
    return all_videos

def extract_video_id_from_filename(filename):
    """Legacy stub — video IDs come from folder/file names directly."""
    base = os.path.splitext(filename)[0]
    if "_frame_" in base:
        return base.split("_frame_")[0]
    return re.sub(r"_?\d+$", "", base) or base

def load_frames_for_video(video_info, frame_interval=1):
    """
    Load pre-extracted frame images from disk and base64-encode them.
    """
    crime_dir  = video_info["crime_dir"]
    frame_files = video_info["frames"]
    video_id   = video_info["video_id"]

    print(f"\nLoading frames for {video_id} from: {crime_dir}")
    print(f"  Total available: {len(frame_files)} | sampling every {frame_interval}")

    IMAGE_EXTS = {".png", ".jpg", ".jpeg", ".bmp"}

    def _frame_num(fname):
        nums = re.findall(r"\d+", fname)
        return int(nums[-1]) if nums else 0

    sorted_files = sorted(frame_files, key=_frame_num)
    selected     = sorted_files[::frame_interval]
    print(f"  Frames to load: {len(selected)}")

    frames_data = {}
    for idx, fname in enumerate(selected):
        if os.path.splitext(fname.lower())[1] not in IMAGE_EXTS:
            continue
        fpath = os.path.join(crime_dir, fname)
        try:
            with open(fpath, "rb") as fh:
                frames_data[fname] = base64.b64encode(fh.read()).decode("utf-8")
            if idx < 3 or idx % 20 == 0 or idx == len(selected) - 1:
                print(f"  [{idx+1:>5}] {fname} ({os.path.getsize(fpath)/1024:.1f} KB)")
        except Exception as e:
            print(f"  Error loading {fname}: {e}")

    print(f"  Done: {len(frames_data)} frames loaded")
    return frames_data

def process_all_crime_folders():
    """Process ALL crime folders with sequential prompting"""
    analyzer = SequentialPromptingAnalyzer()

    all_videos = discover_all_videos_and_frames(FRAMES_DIR)

    if not all_videos:
        print("No videos found to process!")
        return {}

    all_results = {}
    skipped_videos = []

    print(f"\n🔥 PROCESSING ALL {len(all_videos)} VIDEOS WITH SEQUENTIAL PROMPTING 🔥")
    print(f"Using model: {analyzer.model_name}")
    print(f"Frame processing: {'ALL frames' if FRAME_INTERVAL == 1 else f'Every {FRAME_INTERVAL}th frame'}")
    print("🎯 Sequential Mode: Building analysis step by step!")
    print("📁 ENTIRE FOLDER STRUCTURE WILL BE PROCESSED")
    print("="*70)

    checkpoint_lock = threading.Lock()
    print_lock      = threading.Lock()

    def _process_one_video(video_key, video_info):
        with print_lock:
            print(f"\nProcessing video: {video_key}")
            print(f"  Crime type: {video_info['crime_type']}")
            print(f"  Video ID: {video_info['video_id']}")
            print(f"  Frames available: {len(video_info['frames'])}")
        try:
            frames_data = load_frames_for_video(video_info, frame_interval=FRAME_INTERVAL)
            if not frames_data:
                with print_lock:
                    print(f"  No frames loaded for video {video_key} - skipping")
                return video_key, None, "no frames loaded"

            if analyzer._ckpt.is_video_complete(video_key):
                with print_lock:
                    print(f"  [CHECKPOINT] {video_key} already complete, loading from disk")
                cached = analyzer._ckpt.get_video_result(video_key)
                with checkpoint_lock:
                    all_results[video_key] = cached
                    analyzer._comp.record_video(len(frames_data))
                return video_key, cached, None

            results = analyzer.analyze_frames(
                frames_data, video_info['video_id'], video_info['crime_type']
            )
            with checkpoint_lock:
                all_results[video_key] = results
            with print_lock:
                print(f"  Successfully processed {video_key}")
            return video_key, results, None

        except Exception as e:
            with print_lock:
                print(f"  Error processing video {video_key}: {e}")
            return video_key, None, f"error: {e}"

    print(f"\n  Launching ThreadPoolExecutor with {MAX_WORKERS} parallel workers ...")
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = [
            executor.submit(_process_one_video, k, v)
            for k, v in all_videos.items()
        ]
        for fut in as_completed(futures):
            vkey, _res, err = fut.result()
            if err:
                skipped_videos.append(f"{vkey} ({err})")

    summary_file = os.path.join(SAVE_DIR, f"sequential_prompting_summary_{time.strftime('%Y%m%d_%H%M%S')}.json")
    with open(summary_file, 'w') as f:
        json.dump(all_results, f, indent=2)

    if skipped_videos:
        skipped_file = os.path.join(SAVE_DIR, f"skipped_videos_{time.strftime('%Y%m%d_%H%M%S')}.txt")
        with open(skipped_file, 'w') as f:
            f.write("Videos that could not be processed:\n")
            for video in skipped_videos:
                f.write(f"{video}\n")
        print(f"\nSkipped {len(skipped_videos)} videos. List saved to: {skipped_file}")

    print(f"\nComplete sequential analysis saved to: {summary_file}")
    print(f"Successfully processed {len(all_results)} videos out of {len(all_videos)} total")

    return all_results

def test_gemini_api():
    """Test Gemini API connection"""
    print("\nTesting Gemini API connection via Gemini Developer API...")
    try:
        client = genai.Client(
            api_key=os.environ["GOOGLE_API_KEY"],
        )
        response = client.models.generate_content(
            model="gemini-3.1-pro-preview",
            contents="Hello, respond with 'API connection successful'"
        )
        if response.text:
            print(f"✓ Gemini API connection successful!")
            print(f"Response: {response.text[:100]}")
            return True
        else:
            print("✗ No response text received")
            return False
    except Exception as e:
        print(f"✗ API connection failed: {e}")
        return False

def run():
    """Main execution function"""
    print("Sequential Prompting Crime Video Analysis with Gemini - ENTIRE FOLDER PROCESSING")
    print("="*70)
    print("🎯 SEQUENTIAL TECHNIQUE: Building analysis step by step through guided prompts!")
    print("📁 PROCESSES ALL VIDEOS IN ALL CRIME TYPE FOLDERS")
    print("="*70)

    print("Testing directory access...")
    for path in [DATA_DIR, SAVE_DIR]:
        print(f"Path: {path}")
        print(f"  Exists: {os.path.exists(path)}")
        if os.path.exists(path):
            try:
                contents = os.listdir(path)
                print(f"  Contains {len(contents)} items")
                if contents:
                    print(f"  First few items: {contents[:3]}")
            except Exception as e:
                print(f"  Error accessing contents: {str(e)}")

    # Using Gemini Developer API key from file
    print(f"GOOGLE_API_KEY set: {bool(os.environ.get('GOOGLE_API_KEY'))}")

    if not test_gemini_api():
        print("✗ Gemini API test failed. Please check your API key and connection.")
        return

    print("\nVerifying directories:")
    print(f"Data directory exists: {os.path.exists(DATA_DIR)}")
    print(f"Save directory exists: {os.path.exists(SAVE_DIR)}")

    if not os.path.exists(FRAMES_DIR):
        print(f"✗ Frames directory not found: {FRAMES_DIR}")
        return

    os.makedirs(SAVE_DIR, exist_ok=True)

    print("\n🚀 STARTING COMPLETE FOLDER PROCESSING WITH SEQUENTIAL PROMPTING...")
    results = process_all_crime_folders()

    if 'analyzer' in dir():
        _report = analyzer._comp.report()
    else:
        _report = {"note": "analyzer not in scope"}
    _rpath = os.path.join(SAVE_DIR, f"neurips_compute_report_{time.strftime('%Y%m%d_%H%M%S')}.json")
    with open(_rpath, 'w') as _rf:
        json.dump(_report, _rf, indent=2)
    print(f"NeurIPS compute report saved: {_rpath}")

    total_frames_processed = 0
    total_videos_processed = len(results)

    for video_id, video_results in results.items():
        if video_results and 'frames_used' in video_results:
            total_frames_processed += video_results.get('frames_used', 0)

    print("\n" + "="*70)
    print(f"🎉 COMPLETE SEQUENTIAL PROCESSING FINISHED!")
    print(f"Videos processed: {total_videos_processed}")
    print(f"Total frames analyzed: {total_frames_processed}")
    print(f"Model used: {analyzer.model_name if 'analyzer' in locals() else 'gemini-3.1-pro-preview'}")
    print("📁 ENTIRE FOLDER STRUCTURE WAS PROCESSED")
    print("🎯 Sequential prompting technique applied to all videos")
    print("="*70)

run()


Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


Sequential Prompting Crime Video Analysis with Gemini - ENTIRE FOLDER PROCESSING
🎯 SEQUENTIAL TECHNIQUE: Building analysis step by step through guided prompts!
📁 PROCESSES ALL VIDEOS IN ALL CRIME TYPE FOLDERS
Testing directory access...
Path: C:\Opeyemi\PROMPTS\UCF-Data
  Exists: True
  Contains 28 items
  First few items: ['.DS_Store', '._.DS_Store', '._Abuse']
Path: C:\Opeyemi\PROMPTS\RESULTS\GEMINI\SEQUENTIAL
  Exists: True
  Contains 5328 items
  First few items: ['Abuse_Abuse001_x264_sequential_complete_20260428_223425.json', 'Abuse_Abuse001_x264_sequential_step1_20260428_223530.json', 'Abuse_Abuse001_x264_sequential_step2_20260428_223640.json']
GOOGLE_API_KEY set: True

Testing Gemini API connection via Gemini Developer API...


Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


✓ Gemini API connection successful!
Response: API connection successful

Verifying directories:
Data directory exists: True
Save directory exists: True

🚀 STARTING COMPLETE FOLDER PROCESSING WITH SEQUENTIAL PROMPTING...

=== DISCOVERING PRE-EXTRACTED FRAMES ===
Scanning: C:\\Opeyemi\\PROMPTS\\FRAMES
Crime-type folders to process (filtered to 'Stealing'): ['Stealing']
  Stealing/Stealing002_x264: 117 frames
  Stealing/Stealing003_x264: 119 frames
  Stealing/Stealing004_x264: 222 frames
  Stealing/Stealing006_x264: 107 frames
  Stealing/Stealing007_x264: 103 frames
  Stealing/Stealing008_x264: 167 frames
  Stealing/Stealing009_x264: 54 frames
  Stealing/Stealing010_x264: 101 frames
  Stealing/Stealing011_x264: 124 frames
  Stealing/Stealing012_x264: 90 frames
  Stealing/Stealing013_x264: 308 frames
  Stealing/Stealing014_x264: 112 frames
  Stealing/Stealing015_x264: 60 frames
  Stealing/Stealing016_x264: 82 frames
  Stealing/Stealing017_x264: 63 frames
  Stealing/Stealing018_x264: 68 fra